# SageMaker PySDK v3: AI Inference Recommender Demo (JumpStart Models)

Four independently runnable scenarios showing how to use `sagemaker.serve.ModelBuilder` and
`sagemaker.serve.ai_inference_recommender` (`start_benchmark`, `generate_deployment_recommendations`,
`Workload`) against a JumpStart model:

1. **Scenario A — Benchmark an existing endpoint** — measure throughput / latency on something you already deployed.
2. **Scenario B — Get recommendations + deploy** — let the service explore configurations and pick the best one.
3. **Scenario C — Deploy from a prior recommendation job** — re-deploy a config from a job another process ran, via `ModelBuilder.from_recommendation_job`.
4. **Scenario D — Compare LMI vs vLLM and deploy the winner** — run both frameworks' recommendation jobs in parallel and deploy whichever wins on throughput.

Each scenario has three cells: setup (creates resources for the demo) → run (the call you're learning) → cleanup (deletes the resources). Expected output blocks below each run cell are taken from a verified run.

**Cost estimate:** ~$2-3 of `ml.g5.2xlarge` time for the full notebook.

**Prereqs:**
- AWS account with SageMaker quota for `ml.g5.2xlarge` (~$1.50/hr).
- IAM role with `AmazonSageMakerFullAccess` plus S3 read/write.
- Either: SageMaker Studio / notebook instance (ambient role auto-resolved), or local laptop with `SAGEMAKER_ROLE_ARN` env var set.

## Setup

Install the official SageMaker Python SDK v3 from PyPI, then resolve the IAM role and shared config.

`pip install --upgrade sagemaker` installs the top-level `sagemaker` package, which pulls in
`sagemaker-core`, `sagemaker-serve`, and `sagemaker-train` as dependencies — no local wheels needed.

In [ ]:
%pip install --upgrade -q sagemaker

In [ ]:
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Prefer SAGEMAKER_ROLE_ARN env var if set (works on any host); fall back to
# get_execution_role() which only works inside SageMaker Studio / notebook.
ROLE = os.environ.get("SAGEMAKER_ROLE_ARN")
if ROLE:
    print(f"Using SAGEMAKER_ROLE_ARN from env: {ROLE}")
else:
    ROLE = get_execution_role(sagemaker_session=Session())
    print(f"Using ambient SageMaker execution role: {ROLE}")

# Stream INFO logs to stdout so progress is visible in cell output.
import logging, sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("demo")


In [2]:
# Shared config used by all three scenarios.
MODEL_ID = "huggingface-reasoning-qwen3-06b"
INSTANCE_TYPE = "ml.g6.2xlarge"


---

## Scenario A — Benchmark an existing endpoint

**Story:** "I have a SageMaker endpoint serving an LLM. How fast is it?"

The setup cell deploys a fresh JumpStart endpoint to give us something to benchmark — in real use you'd already have one. The run cell calls `start_benchmark` against that endpoint and reads the parsed metrics. Cleanup deletes the endpoint, model, and benchmark job.

### Setup

In [ ]:
log.info(">>> Scenario A SETUP: deploying JumpStart endpoint to bench against")
import time, uuid

from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.serve import ModelBuilder
from sagemaker.train.configs import Compute
from sagemaker.serve import InferenceFramework, PerformanceTarget

uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
ep_name = f"demo-bench-ep-{uid}"
model_name = f"demo-bench-model-{uid}"

mb = ModelBuilder.from_jumpstart_config(
    jumpstart_config=JumpStartConfig(model_id=MODEL_ID),
    compute=Compute(instance_type=INSTANCE_TYPE),
    role_arn=ROLE,
)
core_model = mb.build(model_name=model_name)
core_endpoint = mb.deploy(endpoint_name=ep_name)
print(f"Endpoint InService: {core_endpoint.endpoint_name}")


### Run

In [ ]:
log.info(">>> Scenario A RUN: starting benchmark")
from sagemaker.serve import start_benchmark

# Inline workload kwargs are equivalent to passing a Workload.synthetic(...) instance.
job = start_benchmark(
    endpoint=core_endpoint,
    tokenizer="google/gemma-4-e2b-it",
    concurrency=1,
    request_count=10,
    prompt_input_tokens_mean=32,
    output_tokens_mean=32,
    streaming=True,
    role=ROLE,
    wait=True,
)
print(f"Benchmark terminal state: {job.ai_benchmark_job_status}")

result = job.show_result()
print(result)


### Pick specific metrics from `result`

`result.metrics` exposes the well-known AIPerf metrics as typed fields and *every* AIPerf metric via `.all_metrics` / `.get(name)`.

In [ ]:
# Well-known shortcuts (typed; IDE autocomplete works):
print(f"throughput avg: {result.metrics.request_throughput.avg} req/sec")
print(f"TTFT p99:       {result.metrics.time_to_first_token.p99} ms")
print(f"E2E p90:        {result.metrics.request_latency.p90} ms")

# Any metric AIPerf produced, by raw key:
ott = result.metrics.get("output_token_throughput")
if ott:
    print(f"output_token_throughput p90: {ott.p90} {ott.unit}")


#### Print results as a dataframe

In [9]:
import pandas as pd

result = job.show_result()

rows = []
for name, m in result.metrics.all_metrics.items():
    rows.append({
        "metric":  name,
        "unit":    m.unit,
        "avg":     m.avg,
        "p50":     m.p50,
        "p90":     m.p90,
        "p99":     m.p99,
    })

df = pd.DataFrame(rows).set_index("metric")
print(df)

In [ ]:
df = pd.DataFrame([{
    "throughput_avg_rps":  result.metrics.request_throughput.avg,
    "ttft_p99_ms":         result.metrics.time_to_first_token.p99,
    "e2e_p90_ms":          result.metrics.request_latency.p90,
    "itl_p90_ms":          result.metrics.inter_token_latency.p90 if result.metrics.get("inter_token_latency") else None,
    "output_tok_tput_avg": (result.metrics.get("output_token_throughput") or type('', (), {"avg": None})()).avg,
}])
print(df)

### Cleanup

In [ ]:
log.info(">>> Scenario A CLEANUP")
from sagemaker.core.resources import AIBenchmarkJob, AIWorkloadConfig, EndpointConfig

def _try(label, fn):
    try: fn()
    except Exception as e: print(f"  [skip] {label}: {e}")

# Endpoint, model, endpoint config (deploy uses ep_name for both).
_try(f"endpoint {ep_name}", core_endpoint.delete)
_try(f"model {model_name}", core_model.delete)
_try(f"endpoint config {ep_name}",
     lambda: EndpointConfig.get(endpoint_config_name=ep_name).delete())

# Benchmark job + the workload config it created (name is on the job).
_try(f"benchmark job {job.get_name()}", job.delete)
_try(f"workload config {job.ai_workload_config_identifier}",
     lambda: AIWorkloadConfig.get(
         ai_workload_config_name=job.ai_workload_config_identifier
     ).delete())


---

## Scenario B — Get recommendations + deploy

**Story:** "I have a model. What's the best config to deploy it on?"

The setup cell builds a `ModelBuilder` over a JumpStart model. *(In a real workflow this would be `ModelBuilder(model=trainer_job)` after a training run.)* The run cell calls `mb.generate_deployment_recommendations` to get ranked configurations, then `mb.deploy` to deploy the top one. Cleanup deletes the deployed endpoint and source model.

### Setup

In [ ]:
log.info(">>> Scenario B SETUP: building source model")
import time, uuid

from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.serve import ModelBuilder
from sagemaker.train.configs import Compute
from sagemaker.serve import InferenceFramework, PerformanceTarget

uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
src_model_name = f"demo-rec-source-{uid}"
rec_ep_name    = f"demo-rec-ep-{uid}"

mb = ModelBuilder.from_jumpstart_config(
    jumpstart_config=JumpStartConfig(model_id=MODEL_ID),
    compute=Compute(instance_type=INSTANCE_TYPE),
    role_arn=ROLE,
)
source_model = mb.build(model_name=src_model_name)
print(f"Source model: {source_model.model_name}")


### Run

In [ ]:
log.info(">>> Scenario B RUN: generate_deployment_recommendations + deploy top")
rec_job = mb.generate_deployment_recommendations(
    tokenizer="google/gemma-4-e2b-it",
    concurrency=1,
    request_count=10,
    prompt_input_tokens_mean=32,
    output_tokens_mean=32,
    streaming=True,
    performance_target=PerformanceTarget.TTFT_MS,
    instance_types=[INSTANCE_TYPE],
    advanced_optimization=False,
    framework=InferenceFramework.LMI,
    role_arn=ROLE,
    wait=True,
)

In [ ]:
print(f"Recommendation terminal state: {rec_job.ai_recommendation_job_status}")

# Comparative view across all returned recommendations:
print(mb.recommendations)

# .best is the top-ranked row (== mb.recommendations[0])
top = mb.recommendations.best
print(top)

# Typed access for headline metrics
print(f"throughput avg: {top.expected_performance.request_throughput.avg}")
print(f"TTFT p99:       {top.expected_performance.time_to_first_token.p99}")

In [16]:
import pandas as pd

rows = []
for i, rec in enumerate(mb.recommendations):
    raw  = rec._raw
    spec = raw.model_details.inference_specification_name
    for m in raw.expected_performance:
        rows.append({
            "rank":      i,
            "spec_name": spec,
            "instance":  raw.deployment_configuration.instance_type,
            "metric":    m.metric,   # ← attribute, not subscript
            "stat":      m.stat,
            "value":     float(m.value),
            "unit":      m.unit,
        })

df_long = pd.DataFrame(rows)
print(df_long)

In [ ]:
# auto_approve=True: the recommendation job creates a ModelPackage with approval
# status None — this bypasses the approval check so deploy proceeds immediately.
rec_endpoint = mb.deploy(endpoint_name=rec_ep_name, role=ROLE, wait=True, auto_approve=True)
print(f"\nDeployed: {rec_endpoint.endpoint_name} ({rec_endpoint.endpoint_status})")

### Cleanup

In [ ]:
log.info(">>> Scenario B CLEANUP")
from sagemaker.core.resources import AIRecommendationJob, AIWorkloadConfig, EndpointConfig, Model

def _try(label, fn):
    try: fn()
    except Exception as e: print(f"  [skip] {label}: {e}")

# Endpoint, then the auto-created model + endpoint config from deploy.
ep_cfg_name = rec_endpoint.endpoint_config_name
ep_cfg = EndpointConfig.get(endpoint_config_name=ep_cfg_name)
deployed_model = Model.get(model_name=ep_cfg.production_variants[0].model_name)
_try(f"endpoint {rec_ep_name}", rec_endpoint.delete)
_try(f"deployed model {deployed_model.model_name}", deployed_model.delete)
_try(f"endpoint config {ep_cfg_name}",
     lambda: EndpointConfig.get(endpoint_config_name=ep_cfg_name).delete())

# Source model + recommendation job + its workload config (name on the job).
_try(f"source model {src_model_name}", source_model.delete)
_try(f"recommendation job {rec_job.get_name()}", rec_job.delete)
_try(f"workload config {rec_job.ai_workload_config_identifier}",
     lambda: AIWorkloadConfig.get(
         ai_workload_config_name=rec_job.ai_workload_config_identifier
     ).delete())


---

## Scenario C — Deploy from a previously-run recommendation job

**Story:** "Another team-member ran an `AIRecommendationJob` last week. I want to deploy one of its recommendations *now*, in a different process, without re-running the job."

```python
mb = ModelBuilder.from_recommendation_job("my-rec-job-name")
endpoint = mb.deploy(role=ROLE, recommendation_index=0)
# Or pick a row by name instead of index:
endpoint = mb.deploy(role=ROLE, recommendation_spec_name="my-named-variant")
```

Pin by name (`recommendation_spec_name="high-otps-on-g5-2xlarge"`) when you want the *specific recipe* you validated — the name survives the service re-ordering rows on a re-run. Use `mb.recommendations.best` (or `recommendation_index=0`) when you always want the current top-ranked option, whatever recipe that turns out to be.

The setup cell kicks off a quick recommendation job so we have a real job name to replay against. (In a real workflow you'd skip the bootstrap and pass an existing job name straight to `from_recommendation_job(...)`.) The run cell hydrates a fresh builder from that job and deploys.

### Setup

In [ ]:
log.info(">>> Scenario C SETUP: bootstrap rec job to replay")
import time, uuid

from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.serve import ModelBuilder, Workload
from sagemaker.train.configs import Compute
from sagemaker.serve import InferenceFramework, PerformanceTarget

uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
bootstrap_src_name = f"demo-recreplay-bootstrap-src-{uid}"
rec_ep_name = f"demo-recreplay-ep-{uid}"

bootstrap_mb = ModelBuilder.from_jumpstart_config(
    jumpstart_config=JumpStartConfig(model_id=MODEL_ID),
    compute=Compute(instance_type=INSTANCE_TYPE),
    role_arn=ROLE,
)
bootstrap_src_model = bootstrap_mb.build(model_name=bootstrap_src_name)
bootstrap_rec_job = bootstrap_mb.generate_deployment_recommendations(
    tokenizer="google/gemma-4-e2b-it",
    concurrency=1,
    request_count=10,
    prompt_input_tokens_mean=32,
    output_tokens_mean=32,
    streaming=True,
    performance_target=PerformanceTarget.COST,
    advanced_optimization=False,
    framework=InferenceFramework.LMI,
    role_arn=ROLE,
    wait=True,
)
replay_job_name = bootstrap_rec_job.get_name()
print(f"Bootstrapped {replay_job_name}; replay below.")


### Run

In [ ]:
log.info(">>> Scenario C RUN: replay rec job + deploy")
from sagemaker.serve import ModelBuilder
import time, uuid

# ── Configure ────────────────────────────────────────────────────────────────
replay_job_name = globals().get("replay_job_name") or input("Enter the AIRecommendationJob name to replay: ").strip()

if not globals().get("rec_ep_name"):
    uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
    rec_ep_name = f"demo-recreplay-ep-{uid}"

# Priority-ordered fallback instance types — tries each in order if capacity unavailable.
# All are single-GPU instances comparable to ml.g5.2xlarge.
INSTANCE_TYPE_FALLBACKS = [
    "ml.g5.2xlarge",
    "ml.g6.2xlarge",
    "ml.g6e.2xlarge",
    "ml.g4dn.2xlarge",
]
# ─────────────────────────────────────────────────────────────────────────────

mb = ModelBuilder.from_recommendation_job(replay_job_name)
print(f"Loaded {len(mb.recommendations)} recommendations from {replay_job_name}")

print(mb.recommendations)
print(mb.recommendations.best)

# Deploy with InstancePools so SageMaker falls back automatically on capacity errors.
rec_endpoint = mb.deploy(
    endpoint_name=rec_ep_name,
    role=ROLE,
    recommendation_index=0,
    wait=True,
    auto_approve=True,
    instance_type=INSTANCE_TYPE_FALLBACKS[0],
    instance_pools=INSTANCE_TYPE_FALLBACKS,
)
print(f"Deployed: {rec_endpoint.endpoint_name}")

### Cleanup

In [ ]:
log.info(">>> Scenario C CLEANUP")
from sagemaker.core.resources import AIRecommendationJob, AIWorkloadConfig, EndpointConfig, Model

def _try(label, fn):
    try: fn()
    except Exception as e: print(f"  [skip] {label}: {e}")

# Endpoint + auto-created model + endpoint config from deploy.
ep_cfg_name = rec_endpoint.endpoint_config_name
ep_cfg = EndpointConfig.get(endpoint_config_name=ep_cfg_name)
deployed_model = Model.get(model_name=ep_cfg.production_variants[0].model_name)
_try(f"endpoint {rec_ep_name}", rec_endpoint.delete)
_try(f"deployed model {deployed_model.model_name}", deployed_model.delete)
_try(f"endpoint config {ep_cfg_name}",
     lambda: EndpointConfig.get(endpoint_config_name=ep_cfg_name).delete())

# Bootstrap resources (created by the setup cell).
_try(f"bootstrap source model {bootstrap_src_name}", bootstrap_src_model.delete)
_try(f"bootstrap recommendation job {replay_job_name}", bootstrap_rec_job.delete)
_try(f"bootstrap workload config {bootstrap_rec_job.ai_workload_config_identifier}",
     lambda: AIWorkloadConfig.get(
         ai_workload_config_name=bootstrap_rec_job.ai_workload_config_identifier
     ).delete())


---

## Scenario D — Compare LMI vs vLLM and deploy the winner

**Story:** *"I'm trying to pick between LMI and vLLM for serving my model. Which one is faster, and how do I deploy that one?"*

`generate_deployment_recommendations(framework=InferenceFramework.LMI|InferenceFramework.VLLM, ...)` runs the same workload against the chosen inference framework and returns ranked configs for each. The fastest path to a head-to-head: run two recommendation jobs in parallel, compare `mb.recommendations.best`, and deploy from the winning builder.

The setup cell is shared with Scenarios B/C (it just builds a fresh `ModelBuilder` over the same JumpStart model). The run cell drives both rec jobs concurrently with a thread pool, prints the comparative tables, picks the winner by top-row throughput, and deploys.

### Setup

In [ ]:
log.info(">>> Scenario D SETUP: build two ModelBuilders (one per framework)")
import time, uuid

from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.serve import ModelBuilder
from sagemaker.train.configs import Compute
from sagemaker.serve import InferenceFramework, PerformanceTarget

uid = f"{int(time.time())}-{uuid.uuid4().hex[:8]}"
fw_winner_ep_name = f"demo-fw-winner-ep-{uid}"

def build_for(framework):
    """Build a ModelBuilder for the same model so we can run framework-specific rec jobs."""
    bid = f"{int(time.time())}-{uuid.uuid4().hex[:6]}"
    mb = ModelBuilder.from_jumpstart_config(
        jumpstart_config=JumpStartConfig(model_id=MODEL_ID),
        compute=Compute(instance_type=INSTANCE_TYPE),
        role_arn=ROLE,
    )
    mb.build(model_name=f"demo-fw-{framework.lower()}-{bid}")
    return mb

mb_lmi = build_for(InferenceFramework.LMI)
mb_vllm = build_for(InferenceFramework.VLLM)
print(f"Built source models for LMI ({mb_lmi.built_model.model_name}) and vLLM ({mb_vllm.built_model.model_name}).")

### Run

In [ ]:
log.info(">>> Scenario D RUN: parallel rec jobs (LMI + vLLM)")
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_rec(mb, framework):
    return mb.generate_deployment_recommendations(
        tokenizer="google/gemma-4-e2b-it",
        concurrency=1,
        request_count=10,
        prompt_input_tokens_mean=32,
        output_tokens_mean=32,
        streaming=True,
        performance_target=PerformanceTarget.TTFT_MS,
        instance_types=[INSTANCE_TYPE],
        advanced_optimization=False,
        framework=framework,
        role_arn=ROLE,
        wait=True,
    )

with ThreadPoolExecutor(max_workers=2) as ex:
    futures = {
        ex.submit(run_rec, mb_lmi, "LMI"): InferenceFramework.LMI,
        ex.submit(run_rec, mb_vllm, "VLLM"): InferenceFramework.VLLM,
    }
    for fut in as_completed(futures):
        framework = futures[fut]
        fut.result()
        print(f"[{framework}] rec job complete.")

print("\n=== LMI ===")
print(mb_lmi.recommendations)
print("\n=== vLLM ===")
print(mb_vllm.recommendations)

# Pick the framework whose top recommendation has the higher throughput.
candidates = [(InferenceFramework.LMI, mb_lmi), (InferenceFramework.VLLM, mb_vllm)]
winner_fw, winner_mb = max(
    candidates,
    key=lambda x: x[1].recommendations.best.expected_performance.request_throughput.avg or 0,
)
print(f"\nWinner by throughput: {winner_fw}  recipe={winner_mb.recommendations.best.recommendation_spec_name}")

# auto_approve=True: bypasses ModelPackage approval status check (status=None
# on recommendation-created packages).
fw_endpoint = winner_mb.deploy(
    endpoint_name=fw_winner_ep_name,
    role=ROLE,
    wait=True,
    auto_approve=True,
)
print(f"Deployed: {fw_endpoint.endpoint_name} ({fw_endpoint.endpoint_status})")

### Cleanup

In [ ]:
log.info(">>> Scenario D CLEANUP")
from sagemaker.core.resources import EndpointConfig, Model

def _try(label, fn):
    try: fn()
    except Exception as e: print(f"  [skip] {label}: {e}")

# Winner endpoint + auto-created Model + EndpointConfig (only if deploy succeeded)
if "fw_endpoint" in dir():
    ep_cfg_name = fw_endpoint.endpoint_config_name
    ep_cfg = EndpointConfig.get(endpoint_config_name=ep_cfg_name)
    deployed_model_name = ep_cfg.production_variants[0].model_name
    _try(f"endpoint {fw_winner_ep_name}", fw_endpoint.delete)
    _try(f"deployed model {deployed_model_name}", lambda: Model.get(model_name=deployed_model_name).delete())
    _try(f"endpoint config {ep_cfg_name}", ep_cfg.delete)

# Per-framework source models + rec jobs + workload configs
from sagemaker.core.resources import AIWorkloadConfig
for mb in (mb_lmi, mb_vllm):
    if mb.built_model is not None:
        _try(f"source model {mb.built_model.model_name}", mb.built_model.delete)
    rj = getattr(mb, "_recommendation_job", None)
    if rj is not None:
        _try(f"recommendation job {rj.get_name()}", rj.delete)
        wl_id = getattr(rj, "ai_workload_config_identifier", None)
        if wl_id:
            _try(f"workload config {wl_id}",
                 lambda wl=wl_id: AIWorkloadConfig.get(ai_workload_config_name=wl).delete())


---

## What's next

- **Custom dataset workloads:** `Workload.from_dataset(s3_uri=..., custom_dataset_type="openai-chat", ...)` drives the benchmark from a real request trace. Required for `advanced_optimization=True` + `performance_target=PerformanceTarget.THROUGHPUT`. See `SCENARIOS.md` for the full flow.
- **Reading specific metrics:** `result.metrics.get("any_aiperf_key").p99` works for every metric AIPerf reports; `.all_metrics` gives you the full set.
- **Reading recommendation metrics:** `mb.recommendations.best.expected_performance.request_throughput.avg` (typed) or `.get("RequestThroughput")` (raw service name). Use `mb.recommendations` directly to print a comparative table across rows.
- **Pinning a recommendation by name:** use `recommendation_spec_name="..."` to lock the specific recipe you validated; use `mb.recommendations.best` when you always want the current top-ranked option.